![image_1781002203053.png](./image_1781002203053.png "image_1781002203053.png")

![image_1781002215872.png](./image_1781002215872.png "image_1781002215872.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("ProductSpend").getOrCreate()

# Create product_spend dataset
product_spend_data = [
    ("Electronics", "Laptop", 1, 2000, "2023-01-15"),
    ("Electronics", "Laptop", 2, 1500, "2023-01-20"),
    ("Electronics", "Phone", 1, 800, "2023-02-01"),
    ("Electronics", "Phone", 3, 900, "2023-02-10"),
    ("Electronics", "Tablet", 2, 400, "2023-01-25"),
    ("Clothing", "Jacket", 1, 300, "2023-01-12"),
    ("Clothing", "Jacket", 3, 350, "2023-02-05"),
    ("Clothing", "Jeans", 2, 200, "2023-01-18"),
    ("Clothing", "Jeans", 1, 250, "2023-02-15"),
    ("Clothing", "Sneakers", 3, 150, "2023-01-30")
]

product_spend_columns = ["category", "product", "user_id", "spend", "transaction_date"]

product_spend_df = spark.createDataFrame(product_spend_data, product_spend_columns)

# Show the DataFrame
product_spend_df.show()


In [0]:
result_df = (
    product_spend_df.groupBy("category", "product")
    .agg(f.sum("spend").alias("total_spend"))
    .withColumn(
        "rn",
        f.dense_rank().over(
            Window.partitionBy("category").orderBy(f.desc("total_spend"))
        ),
    )
    .filter((f.col("rn") == 1) | (f.col("rn") == 2))
    .select(f.col("category"), f.col("product"), f.col("total_spend"))
)
display(result_df)